In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error
import pickle

In [ ]:
def read_dataframe(path):
    df=pd.read_parquet(path)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    
    df['duration'] = df.lpep_dropoff_datetime-df.lpep_pickup_datetime
    
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    df=df[((df.duration>=1) & (df.duration<=60))]
    
    categorical = ['PULocationID', 'DOLocationID']
    
    df[categorical]=df[categorical].astype(str)

    return df

In [3]:
df_train=read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-01.parquet')
df_val=read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-02.parquet')

In [4]:
len(df_train), len(df_val)

(38088, 35319)

In [5]:
categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv=DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [6]:
target='duration'
y_train= df_train[target].values
y_val= df_val[target].values

In [7]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

8.337009069792204

In [14]:
X_train.indices = X_train.indices.astype('int32')
X_train.indptr = X_train.indptr.astype('int32')

X_val.indices = X_val.indices.astype('int32')
X_val.indptr = X_val.indptr.astype('int32')

lr = Lasso(0.1)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

9.86069501503534

In [ ]:
with open('lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv,lr), f_out)